# Notebook 02-UNSW — Train Six Canonical Models on UNSW-NB15

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection  
**Author:** Md Anas Biswas, University of Portsmouth

## What this notebook does

Trains the same six canonical models used on NSL-KDD and CIC-IDS2017:

| Model | Target | Imbalance strategy |
|---|---|---|
| rf_binary_cw  | binary  | class_weight='balanced' |
| xgb_binary_cw | binary  | scale_pos_weight |
| dnn_binary_cw | binary  | class-weighted BCE |
| rf_5class_smote  | 5-class | SMOTE |
| xgb_5class_smote | 5-class | SMOTE |
| dnn_5class_smote | 5-class | SMOTE + class-weighted CE |

Hyperparameters are deliberately **identical** to the NSL-KDD and CIC-IDS2017 training notebooks so cross-dataset comparisons in Notebook 06-UNSW (Krishna agreement) and the paper aren't confounded by hyperparameter differences.

## Outputs

Saved to `models/unsw_nb15/`:
- 6 model files (RF/XGB as `.joblib`, DNN as `.pt`)
- 6 prediction `.npy` files (hard labels)
- 6 probability `.npy` files (softmax/sigmoid outputs)
- `metrics.json` — one dict per model, same key names as NSL-KDD/CIC versions

Saved to `results/tables/`:
- `unsw_model_comparison.csv` — comparison table

Saved to `results/figures/`:
- `unsw_confusion_matrices.png` — 6-panel confusion matrices

## Time budget (T4 GPU)

- RF × 2 ≈ 4–6 min total
- XGBoost × 2 ≈ 2–4 min total (tree_method='hist')
- DNN × 2 ≈ 10–15 min total
- Total ≈ 20–30 min

Models are saved immediately after each one trains, so a runtime disconnect partway through doesn't lose finished work.

## 1. Environment setup (persists across runtime resets)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import shutil

# Restore git identity from Drive — survives Colab runtime resets
shutil.copy('/content/drive/MyDrive/XIDS_Research/.gitconfig',       '/root/.gitconfig')
shutil.copy('/content/drive/MyDrive/XIDS_Research/.git-credentials', '/root/.git-credentials')

PROJECT_DIR = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(PROJECT_DIR)

!git config --global credential.helper 'store --file /content/drive/MyDrive/XIDS_Research/.git-credentials'
!git pull origin main

# Verify
!git config --global --get user.name
!git config --global --get user.email

From https://github.com/anasbiswas1/xids-research
 * branch            main       -> FETCH_HEAD
Already up to date.
Md Anas Biswas
anasbiswas@gmail.com


In [3]:
# GPU check
!nvidia-smi -L

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nPyTorch device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

GPU 0: Tesla T4 (UUID: GPU-9befd585-894b-562f-6a6d-14739f390c27)

PyTorch device: cuda
GPU: Tesla T4
Memory: 14.6 GB


## 2. Load preprocessed UNSW-NB15 arrays

In [4]:
import numpy as np
import json
from pathlib import Path

REPO      = Path(PROJECT_DIR)
PROC_DIR  = REPO / 'data/processed/unsw_nb15'
MODELS_DIR = REPO / 'models/unsw_nb15'
PREDS_DIR  = MODELS_DIR / 'predictions'
PROBS_DIR  = MODELS_DIR / 'probabilities'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)
PROBS_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(PROC_DIR / 'X_train.npy')
X_test  = np.load(PROC_DIR / 'X_test.npy')
y_train_binary = np.load(PROC_DIR / 'y_train_binary.npy')
y_test_binary  = np.load(PROC_DIR / 'y_test_binary.npy')
y_train_5class = np.load(PROC_DIR / 'y_train_5class.npy')
y_test_5class  = np.load(PROC_DIR / 'y_test_5class.npy')

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)
with open(PROC_DIR / 'class_mappings.json') as f:
    class_mappings = json.load(f)

FIVE_CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
N_FEATURES = X_train.shape[1]

print(f'X_train: {X_train.shape}  dtype={X_train.dtype}')
print(f'X_test:  {X_test.shape}')
print(f'Binary train positives: {y_train_binary.sum():,} / {len(y_train_binary):,}')
print(f'5-class train distribution:')
for cid, name in enumerate(FIVE_CLASS_NAMES):
    n = int((y_train_5class == cid).sum())
    print(f'  {cid} {name:8s}: {n:>7,}')

X_train: (135341, 196)  dtype=float32
X_test:  (63461, 196)
Binary train positives: 79,341 / 135,341
5-class train distribution:
  0 Normal  :  56,000
  1 DoS     :  12,264
  2 Probe   :  12,491
  3 R2L     :  53,323
  4 U2R     :   1,263


## 3. Shared evaluation helper

Same metric set and key naming as the NSL-KDD and CIC-IDS2017 versions, so downstream notebooks don't need branching logic per dataset.

In [5]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, classification_report, confusion_matrix,
    roc_auc_score
)

ALL_METRICS = {}

def evaluate(model_name, y_true, y_pred, y_proba, target_type):
    """Compute the same metric set used for NSL-KDD and CIC-IDS2017."""
    metrics = {
        'model':        model_name,
        'target_type':  target_type,
        'accuracy':     float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall_macro':    float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_macro':        float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_weighted':     float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'mcc':             float(matthews_corrcoef(y_true, y_pred)),
    }
    if target_type == 'binary':
        # AUC-ROC for binary using the positive-class probability
        pos_proba = y_proba[:, 1] if y_proba.ndim == 2 else y_proba
        metrics['auc_roc'] = float(roc_auc_score(y_true, pos_proba))
        metrics['per_class_f1'] = {
            'Normal': float(f1_score(y_true, y_pred, pos_label=0)),
            'Attack': float(f1_score(y_true, y_pred, pos_label=1)),
        }
    else:
        per_class = f1_score(y_true, y_pred, average=None, labels=range(5), zero_division=0)
        metrics['per_class_f1'] = {name: float(per_class[i]) for i, name in enumerate(FIVE_CLASS_NAMES)}
    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred).tolist()
    return metrics

def save_artifacts(model_name, model, y_pred, y_proba, model_kind='sklearn'):
    """Persist model, predictions, probabilities."""
    import joblib
    if model_kind == 'sklearn':
        joblib.dump(model, MODELS_DIR / f'{model_name}.joblib')
    elif model_kind == 'xgboost':
        joblib.dump(model, MODELS_DIR / f'{model_name}.joblib')
    elif model_kind == 'torch':
        torch.save(model.state_dict(), MODELS_DIR / f'{model_name}.pt')
    np.save(PREDS_DIR / f'{model_name}_preds.npy', y_pred)
    np.save(PROBS_DIR / f'{model_name}_proba.npy', y_proba)

def checkpoint_metrics():
    """Save metrics.json after every model so a disconnect doesn't lose progress."""
    with open(MODELS_DIR / 'metrics.json', 'w') as f:
        json.dump(ALL_METRICS, f, indent=2, default=str)

print('Evaluation helpers ready.')

Evaluation helpers ready.


## 4. Random Forest — binary, class_weight='balanced'

In [6]:
from sklearn.ensemble import RandomForestClassifier
import time

name = 'rf_binary_cw'
print(f'Training {name}...')
t0 = time.time()

clf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
clf.fit(X_train, y_train_binary)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

metrics = evaluate(name, y_test_binary, y_pred, y_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
save_artifacts(name, clf, y_pred, y_proba, 'sklearn')
checkpoint_metrics()

print(f"  Accuracy={metrics['accuracy']:.4f}  F1-macro={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}  AUC={metrics['auc_roc']:.4f}")
print(f"  Time: {metrics['train_time_sec']}s")

Training rf_binary_cw...
  Accuracy=0.8323  F1-macro=0.8322  MCC=0.7021  AUC=0.9661
  Time: 11.8s


## 5. XGBoost — binary, scale_pos_weight

In [7]:
import xgboost as xgb

name = 'xgb_binary_cw'
print(f'Training {name}...')
t0 = time.time()

# scale_pos_weight = negatives / positives  (mirrors class_weight='balanced' for XGBoost)
spw = float((y_train_binary == 0).sum()) / float((y_train_binary == 1).sum())
print(f'  scale_pos_weight = {spw:.3f}')

clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    tree_method='hist',
    device='cuda' if device.type == 'cuda' else 'cpu',
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train_binary)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

metrics = evaluate(name, y_test_binary, y_pred, y_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
save_artifacts(name, clf, y_pred, y_proba, 'xgboost')
checkpoint_metrics()

print(f"  Accuracy={metrics['accuracy']:.4f}  F1-macro={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}  AUC={metrics['auc_roc']:.4f}")
print(f"  Time: {metrics['train_time_sec']}s")

Training xgb_binary_cw...
  scale_pos_weight = 0.706


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [17:40:30] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  Accuracy=0.8529  F1-macro=0.8524  MCC=0.7268  AUC=0.9721
  Time: 3.7s


## 6. DNN architecture (shared for binary and 5-class)

Same architecture used on NSL-KDD and CIC-IDS2017: MLP 256-128-64-32 with BatchNorm and dropout 0.3.

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

class XIDSMLP(nn.Module):
    """256-128-64-32 with BatchNorm, ReLU, dropout 0.3."""
    def __init__(self, n_features, n_classes, p_drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, 64),         nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(64, 32),          nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(32, n_classes),
        )
    def forward(self, x):
        return self.net(x)


def train_dnn(X_tr, y_tr, X_te, y_te, n_classes, class_weights=None,
              epochs=40, batch_size=512, lr=1e-3, patience=5, verbose=True):
    """Train DNN with early stopping on a 90/10 stratified val split."""
    from sklearn.model_selection import train_test_split

    X_tr2, X_val, y_tr2, y_val = train_test_split(
        X_tr, y_tr, test_size=0.1, random_state=42, stratify=y_tr,
    )

    Xt  = torch.tensor(X_tr2, dtype=torch.float32)
    yt  = torch.tensor(y_tr2, dtype=torch.long)
    Xv  = torch.tensor(X_val, dtype=torch.float32).to(device)
    yv  = torch.tensor(y_val, dtype=torch.long).to(device)
    Xte = torch.tensor(X_te,  dtype=torch.float32).to(device)

    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

    model = XIDSMLP(X_tr.shape[1], n_classes).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    if class_weights is not None:
        cw_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=cw_tensor)
    else:
        criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_state    = None
    no_improve    = 0

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xv)
            val_loss   = criterion(val_logits, yv).item()
            val_acc    = (val_logits.argmax(1) == yv).float().mean().item()

        if verbose and (ep == 1 or ep % 5 == 0):
            print(f'  ep {ep:>3d}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state    = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f'  early stop at epoch {ep}')
                break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        logits  = model(Xte)
        y_proba = F.softmax(logits, dim=1).cpu().numpy()
        y_pred  = y_proba.argmax(axis=1)

    return model, y_pred, y_proba, best_val_loss

print('DNN helpers ready.')

DNN helpers ready.


## 7. DNN — binary, class-weighted

In [9]:
from sklearn.utils.class_weight import compute_class_weight

name = 'dnn_binary_cw'
print(f'Training {name}...')
t0 = time.time()

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train_binary)
print(f'  class weights: Normal={cw[0]:.3f}, Attack={cw[1]:.3f}')

model, y_pred, y_proba, _ = train_dnn(
    X_train, y_train_binary, X_test, y_test_binary,
    n_classes=2, class_weights=cw,
    epochs=40, batch_size=512, lr=1e-3, patience=5,
)

metrics = evaluate(name, y_test_binary, y_pred, y_proba, 'binary')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
save_artifacts(name, model, y_pred, y_proba, 'torch')
checkpoint_metrics()

print(f"  Accuracy={metrics['accuracy']:.4f}  F1-macro={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}  AUC={metrics['auc_roc']:.4f}")
print(f"  Time: {metrics['train_time_sec']}s")

Training dnn_binary_cw...
  class weights: Normal=1.208, Attack=0.853
  ep   1  val_loss=0.1687  val_acc=0.9169
  ep   5  val_loss=0.1591  val_acc=0.9227
  ep  10  val_loss=0.1567  val_acc=0.9227
  ep  15  val_loss=0.1518  val_acc=0.9272
  ep  20  val_loss=0.1527  val_acc=0.9269
  early stop at epoch 20
  Accuracy=0.8321  F1-macro=0.8320  MCC=0.6981  AUC=0.9655
  Time: 36.7s


## 8. SMOTE on 5-class training data

We oversample the four attack classes to the largest minority class size, not the Normal class size. Pushing U2R from 1,263 to 56,000 would inject too many synthetic neighbours from too few real seeds. We oversample to the second-largest class (R2L ≈ 53K), keeping the Normal class as-is.

In [11]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print('Pre-SMOTE 5-class distribution:')
pre = Counter(y_train_5class)
for cid in range(5):
    print(f'  {cid} {FIVE_CLASS_NAMES[cid]:8s}: {pre[cid]:>7,}')

# Target sizes: keep Normal as-is, lift each attack class up to the largest attack class
largest_attack = max(pre[c] for c in [1, 2, 3, 4])
sampling_strategy = {
    0: pre[0],                            # Normal kept
    1: max(pre[1], largest_attack),       # DoS up to R2L size
    2: max(pre[2], largest_attack),       # Probe up to R2L size
    3: max(pre[3], largest_attack),       # R2L kept (already largest)
    4: max(pre[4], largest_attack),       # U2R up to R2L size  ← biggest synthesis
}

# SMOTE: n_jobs removed in imbalanced-learn 0.11+; runs single-threaded internally
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=5)
X_train_sm, y_train_5class_sm = smote.fit_resample(X_train, y_train_5class)

X_train_sm = X_train_sm.astype(np.float32)

print('\nPost-SMOTE 5-class distribution:')
post = Counter(y_train_5class_sm)
for cid in range(5):
    print(f'  {cid} {FIVE_CLASS_NAMES[cid]:8s}: {post[cid]:>7,}')
print(f'\nResampled shape: {X_train_sm.shape}')

Pre-SMOTE 5-class distribution:
  0 Normal  :  56,000
  1 DoS     :  12,264
  2 Probe   :  12,491
  3 R2L     :  53,323
  4 U2R     :   1,263

Post-SMOTE 5-class distribution:
  0 Normal  :  56,000
  1 DoS     :  53,323
  2 Probe   :  53,323
  3 R2L     :  53,323
  4 U2R     :  53,323

Resampled shape: (269292, 196)


## 9. Random Forest — 5-class, SMOTE

In [12]:
name = 'rf_5class_smote'
print(f'Training {name}...')
t0 = time.time()

clf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
)
clf.fit(X_train_sm, y_train_5class_sm)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

metrics = evaluate(name, y_test_5class, y_pred, y_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
save_artifacts(name, clf, y_pred, y_proba, 'sklearn')
checkpoint_metrics()

print(f"  Accuracy={metrics['accuracy']:.4f}  F1-macro={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")
print(f"  Per-class F1: {metrics['per_class_f1']}")
print(f"  Time: {metrics['train_time_sec']}s")

Training rf_5class_smote...
  Accuracy=0.6935  F1-macro=0.5261  MCC=0.5280
  Per-class F1: {'Normal': 0.8374353925670687, 'DoS': 0.31080703714046354, 'Probe': 0.5441441441441441, 'R2L': 0.5926547536775595, 'U2R': 0.3455083909180652}
  Time: 36.2s


## 10. XGBoost — 5-class, SMOTE

In [13]:
name = 'xgb_5class_smote'
print(f'Training {name}...')
t0 = time.time()

clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    tree_method='hist',
    device='cuda' if device.type == 'cuda' else 'cpu',
    objective='multi:softprob',
    num_class=5,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train_sm, y_train_5class_sm)

y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

metrics = evaluate(name, y_test_5class, y_pred, y_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
save_artifacts(name, clf, y_pred, y_proba, 'xgboost')
checkpoint_metrics()

print(f"  Accuracy={metrics['accuracy']:.4f}  F1-macro={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")
print(f"  Per-class F1: {metrics['per_class_f1']}")
print(f"  Time: {metrics['train_time_sec']}s")

Training xgb_5class_smote...
  Accuracy=0.7144  F1-macro=0.5815  MCC=0.5597
  Per-class F1: {'Normal': 0.8443988170939123, 'DoS': 0.43955369024465146, 'Probe': 0.6073607557567408, 'R2L': 0.6102594930367699, 'U2R': 0.4057649667405765}
  Time: 17.8s


## 11. DNN — 5-class, SMOTE

In [ ]:
name = 'dnn_5class_smote'
print(f'Training {name}...')
t0 = time.time()

# Light class weighting on top of SMOTE — small belt-and-braces lift for the rarest class
cw = compute_class_weight('balanced', classes=np.arange(5), y=y_train_5class_sm)
print(f'  class weights: {dict(zip(FIVE_CLASS_NAMES, cw.round(3)))}')

model, y_pred, y_proba, _ = train_dnn(
    X_train_sm, y_train_5class_sm, X_test, y_test_5class,
    n_classes=5, class_weights=cw,
    epochs=40, batch_size=512, lr=1e-3, patience=5,
)

metrics = evaluate(name, y_test_5class, y_pred, y_proba, 'multiclass')
metrics['train_time_sec'] = round(time.time() - t0, 1)
ALL_METRICS[name] = metrics
save_artifacts(name, model, y_pred, y_proba, 'torch')
checkpoint_metrics()

print(f"  Accuracy={metrics['accuracy']:.4f}  F1-macro={metrics['f1_macro']:.4f}  MCC={metrics['mcc']:.4f}")
print(f"  Per-class F1: {metrics['per_class_f1']}")
print(f"  Time: {metrics['train_time_sec']}s")

Training dnn_5class_smote...
  class weights: {'Normal': np.float64(0.962), 'DoS': np.float64(1.01), 'Probe': np.float64(1.01), 'R2L': np.float64(1.01), 'U2R': np.float64(1.01)}
  ep   1  val_loss=0.7241  val_acc=0.7013
  ep   5  val_loss=0.5664  val_acc=0.7674
  ep  10  val_loss=0.5493  val_acc=0.7722
  ep  15  val_loss=0.5264  val_acc=0.7805
  ep  20  val_loss=0.5141  val_acc=0.7885
  ep  25  val_loss=0.5143  val_acc=0.7868
  ep  30  val_loss=0.5097  val_acc=0.7870


## 12. Comparison table

In [ ]:
import pandas as pd

rows = []
for name, m in ALL_METRICS.items():
    row = {
        'Model':       name,
        'Target':      m['target_type'],
        'Accuracy':    round(m['accuracy'], 4),
        'F1-macro':    round(m['f1_macro'], 4),
        'F1-weighted': round(m['f1_weighted'], 4),
        'MCC':         round(m['mcc'], 4),
    }
    if m['target_type'] == 'binary':
        row['AUC-ROC']  = round(m['auc_roc'], 4)
        row['U2R F1']   = None
    else:
        row['AUC-ROC'] = None
        row['U2R F1']  = round(m['per_class_f1']['U2R'], 4)
    row['Time (s)'] = m['train_time_sec']
    rows.append(row)

comparison = pd.DataFrame(rows)
comparison_path = REPO / 'results/tables/unsw_model_comparison.csv'
comparison.to_csv(comparison_path, index=False)

print(comparison.to_string(index=False))
print(f'\nSaved: {comparison_path}')

## 13. Confusion matrices figure

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
model_order = [
    'rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw',
    'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
]

for ax, name in zip(axes.flat, model_order):
    cm = np.array(ALL_METRICS[name]['confusion_matrix'])
    if ALL_METRICS[name]['target_type'] == 'binary':
        labels = ['Normal', 'Attack']
    else:
        labels = FIVE_CLASS_NAMES

    # Row-normalise for readability
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f"{name}\nAcc={ALL_METRICS[name]['accuracy']:.3f}, F1m={ALL_METRICS[name]['f1_macro']:.3f}", fontsize=10)

    for i in range(len(labels)):
        for j in range(len(labels)):
            colour = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=8, color=colour)

plt.suptitle('UNSW-NB15 — Confusion matrices (row-normalised)', fontsize=14, y=1.00)
plt.tight_layout()

cm_path = REPO / 'results/figures/unsw_confusion_matrices.png'
plt.savefig(cm_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {cm_path}')

## 14. Commit and push

In [ ]:
os.chdir(PROJECT_DIR)
!git add notebooks/02_unsw_train_models.ipynb \
         results/figures/unsw_confusion_matrices.png \
         results/tables/unsw_model_comparison.csv
!git commit -m "Notebook 02-UNSW: train 6 canonical models (RF/XGB/DNN x binary_cw/5class_smote) on UNSW-NB15"
!git push origin main

---

## What to check before moving on

1. **All six models trained without errors** — comparison table has six rows.
2. **U2R F1 > 0** in the 5-class models. SMOTE-then-RF historically gives the strongest U2R recall.
3. **MCC across binary models** should be in the 0.85+ range (UNSW is easier than NSL-KDD on binary because Normal vs Attack flows have more separated feature distributions).
4. **DNN early stopping** should have triggered before epoch 40 — if every DNN ran the full 40 epochs, val loss isn't converging and we'll need to look at it.
5. **All artifacts saved** — check that `models/unsw_nb15/` contains six model files, six pred files, six proba files, and `metrics.json`.

## Next step

**Notebook 03-UNSW** — per-class isotonic calibration. Mirrors the validated calibration approach used on NSL-KDD (ECE reductions 47-98%) and CIC-IDS2017.